# SmartPolicy — real RAG in Colab
This notebook runs the same Node RAG engine as the website, using a private API key. No plugin is needed. API calls are billable. Do not enter personal or confidential data.

1. In Colab’s **Secrets** (key icon), create `OPENAI_API_KEY` and enable notebook access.
2. Upload the new `SmartPolicy_RAG_Update.zip` when requested.
3. Run the setup and demo cells. The Gradio public URL lasts only while this Colab session runs; it does not configure the existing hosted website.
4. A missing key causes an error, not a fake LLM demonstration.


In [ ]:
import os, pathlib, zipfile, subprocess, shutil
from google.colab import files, userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
if not os.environ['OPENAI_API_KEY'].strip():
    raise RuntimeError('Add OPENAI_API_KEY to Colab Secrets first.')
os.environ['LLM_ENABLED'] = 'true'
os.environ['OPENAI_MODEL'] = 'gpt-4o'
uploaded = files.upload()
zip_names = [n for n in uploaded if n.endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError('Upload only SmartPolicy_RAG_Update.zip.')
destination = pathlib.Path('/content/smartpolicy-rag')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as z:
    for item in z.infolist():
        target = (destination / item.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise RuntimeError('Unsafe ZIP path.')
    z.extractall(destination)
roots = [p.parent for p in destination.rglob('package.json') if (p.parent / 'server/rag.mjs').exists()]
if len(roots) != 1:
    raise RuntimeError('Could not locate the RAG project.')
project = roots[0]
if not shutil.which('node'):
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nodejs'], check=True)
major = int(subprocess.check_output(['node','--version'], text=True).strip().lstrip('v').split('.')[0])
if major < 18:
    raise RuntimeError('Node 18+ is needed; use a current Colab runtime.')
subprocess.run(['node','scripts/build.mjs'], cwd=project, check=True)
subprocess.run(['node','--test','tests/rag.test.mjs'], cwd=project, check=True)
subprocess.run(['pip','-q','install','gradio>=5,<6','requests'],check=True)
server = subprocess.Popen(['node','scripts/serve.mjs'], cwd=project, env=os.environ.copy())
print('Setup complete. The next cell starts the live RAG demo.')


In [ ]:
import time, requests, gradio as gr
for attempt in range(30):
    try:
        health = requests.get('http://localhost:8000/api/status', timeout=2).json()
        if health['rag_configured']:
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError('RAG server did not start.')
def ask(question, history):
    if not question.strip():
        return '', history
    response = requests.post('http://localhost:8000/api/ask',json={'question':question},timeout=120)
    payload = response.json()
    if not response.ok:
        raise gr.Error(payload.get('error','Model call failed.'))
    evidence = '\n\n'.join(f"**{c['title']} — {c['heading']} (PDF page {c['page']})**\n\n{c['text']}" for c in payload['sources'])
    label = f"Live LLM: {payload['model']}" if payload.get('llm_called') else 'Safety guard; no LLM called'
    message = payload['answer'] + '\n\n' + evidence + '\n\n' + label
    return '', (history or []) + [{'role':'user','content':question},{'role':'assistant','content':message}]
with gr.Blocks(title='SmartPolicy RAG') as demo:
    gr.Markdown('# SmartPolicy — Live RAG\nFictional policy data. Questions and excerpts go to OpenAI. Verify citations; no claim approval.')
    chat = gr.Chatbot(type='messages', height=500)
    question = gr.Textbox(label='English policy question', max_lines=4)
    send = gr.Button('Ask')
    clear = gr.Button('Start a new conversation')
    send.click(ask,[question,chat],[question,chat])
    question.submit(ask,[question,chat],[question,chat])
    clear.click(lambda: ('',[]),outputs=[question,chat])
demo.queue().launch(share=True)


## Optional live evaluation (additional API charges)
Run the next cell only if you want the full 30-question evaluation. Outputs are real model records, not independent validation. Ragas adds judge calls and must be run separately as described in the README. Record failures and do not transfer the old BM25 scores to this RAG version.


In [ ]:
# Uncomment to make billable calls for all 30 reused evaluation questions.
# subprocess.run(['node','scripts/evaluate_live.mjs','--all'],cwd=project,env=os.environ.copy(),check=True)
# files.download(str(project/'results/live-rag.json'))
